# Orpheus-3B Sunbird Multilingual Finetune

Finetune `unsloth/orpheus-3b-0.1-pretrained` on the **full** Sunbird/tts corpus — every available language config and every speaker — with MLflow tracking and a final push to `sunbird/orpheus-3b-tts-multilingual`.

**Differences from `Orpheus_3B_Sunbird_Luganda.ipynb`:**
- #2.1 discovers all language configs via `get_dataset_config_names("Sunbird/tts")` and concatenates them into one `ds_train` / `ds_test`. No language or speaker filter.
- #2.2 sets `source = example["speaker_id"]` so every row carries its own speaker tag (per-row, not constant). The model learns the multi-speaker prompt format `f"{speaker_id}: {text}"` across all speakers it sees.
- #2.7 holds aside up to 10 *diverse* test prompts (one per unique speaker) for inference quality eyeballing.
- #3 writes checkpoints to `orpheus-3B/outputs/multilingual/`.
- #5 pushes to `sunbird/orpheus-3b-tts-multilingual`.

**Resource note:** The combined dataset is several GB and tens of thousands of utterances. Expect 4–8 hours of training on an RTX 4090 at `num_train_epochs=3`. The smoke default (`max_steps=60`) still completes in a few minutes for pipeline verification.

**References**: `Orpheus_(3B)_TTS.ipynb` and https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning

## #0 Setup & secrets

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install snac torchcodec "datasets>=3.4.1,<4.0.0"
# `--ignore-installed blinker` works around vast.ai/Debian images where
# blinker is installed via apt (no RECORD file) and pip refuses to upgrade it.
!pip install --ignore-installed blinker mlflow soundfile librosa

In [ ]:
import os
from getpass import getpass

# HF + MLflow credentials (interactive)
os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")
os.environ.setdefault("MLFLOW_TRACKING_URI", "https://mlflow.sunbird.ai/")
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", "orpheus-3b-tts-multilingual-finetuning")
os.environ["MLFLOW_TRACKING_USERNAME"] = getpass("MLFLOW_TRACKING_USERNAME: ")
os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("MLFLOW_TRACKING_PASSWORD: ")

In [ ]:
import locale
from collections import Counter
from pathlib import Path

import torch
import soundfile as sf
from datasets import load_dataset, Audio, concatenate_datasets, get_dataset_config_names
from snac import SNAC
from IPython.display import display, Audio as AudioDisplay

locale.getpreferredencoding = lambda: "UTF-8"

OUTPUT_DIR = Path("orpheus-3B/outputs/multilingual")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "inference_samples").mkdir(parents=True, exist_ok=True)

## #1 Load model + LoRA

In [ ]:
from unsloth import FastLanguageModel

# 4096 tokens ≈ 16 s of audio — covers most Sunbird utterances after
# SNAC tokenisation. Longer rows are dropped by the length filter in #2.6.
# Drop to 2048 if you're on <16 GB VRAM; bump to 8192 if you see many
# rows dropped at the #2.6 filter.
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/orpheus-3b-0.1-pretrained",
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,            # auto-detect bf16 on Ampere+, fp16 on T4
    load_in_4bit = False,    # LoRA 16-bit for higher accuracy on TTS
    token = os.environ["HF_TOKEN"],
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## #2 Data prep — multilingual, multi-speaker

### #2.1 Discover and load all language configs

Pulls every config available on `Sunbird/tts` and concatenates the train / test splits into single datasets. To restrict to a subset, override `CONFIGS` with an explicit list (e.g. `CONFIGS = ["lug", "ach"]`).

In [ ]:
CONFIGS = get_dataset_config_names("Sunbird/tts", token=os.environ.get("HF_TOKEN"))
# Or override: CONFIGS = ["lug", "ach", "nyn"]
print(f"Available language configs: {CONFIGS}")

train_parts, test_parts = [], []
for cfg in CONFIGS:
    print(f"Loading {cfg}...")
    train_parts.append(load_dataset("Sunbird/tts", cfg, split="train"))
    test_parts.append(load_dataset("Sunbird/tts", cfg, split="test"))

ds_train_full = concatenate_datasets(train_parts)
ds_test_full  = concatenate_datasets(test_parts)
ds_train      = ds_train_full   # no speaker filter
ds_test       = ds_test_full

print(f"\ntrain rows: {len(ds_train)}")
print(f"test  rows: {len(ds_test)}")

speaker_counts = Counter(ds_train["speaker_id"])
print(f"\n{len(speaker_counts)} unique speakers in train; top 10 by row count:")
for sid, n in speaker_counts.most_common(10):
    print(f"  {sid}: {n}")

lang_counts = Counter(ds_train["language"])
print(f"\n{len(lang_counts)} unique languages in train:")
for lang, n in lang_counts.most_common():
    print(f"  {lang}: {n}")

### #2.2 Tag every row with `source = speaker_id` (per-row, not constant)

The training prompt becomes `f"{speaker_id}: {text}"`, so the model learns to condition on the speaker tag and produce that speaker's voice at inference time. Unlike the single-speaker notebook (where every row had the same constant tag), here `source` varies per row across the entire speaker pool.

In [ ]:
def add_source(example):
    example["source"] = example["speaker_id"]
    return example

ds_train = ds_train.map(add_source)
ds_test  = ds_test.map(add_source)
ds_train[0].keys()

### #2.3 Cast `audio` to 24 kHz (Orpheus / SNAC expected sample rate)

Sunbird/tts stores audio as `{"bytes": <ogg-vorbis>, "path": ...}` (decode=False). Casting to `Audio(sampling_rate=24000)` switches the column to lazy-decode mode — on access `row["audio"]` returns `{"array": np.ndarray, "sampling_rate": 24000, "path": str}`, which is what `add_codes` consumes in #2.4.

In [ ]:
ds_train = ds_train.cast_column("audio", Audio(sampling_rate=24000))
ds_test  = ds_test.cast_column("audio",  Audio(sampling_rate=24000))
# Sanity check — should print 24000 (and KeyError if the cast didn't take).
ds_train[0]["audio"]["sampling_rate"]

### #2.4 SNAC tokenisation (24 kHz, 7 codes per frame, per-layer offsets)

Same SNAC encoding as the single-speaker notebook. With tens of thousands of utterances this step is the slowest part of data prep — 30–90 minutes on an RTX 4090. HF `datasets.map` caches the result to disk, so re-running the notebook skips re-encoding.

**Pre-filter**: before encoding, drop any row whose tokenised text alone exceeds `MAX_SEQ_LEN`. Such a row could never fit in context (audio codes only push it further over), so SNAC-encoding it would be wasted work.

In [ ]:
# Pre-filter overlong text rows BEFORE the expensive SNAC pass.
# A row whose text alone tokenises to > MAX_SEQ_LEN cannot fit even
# without audio codes; the #2.6 length filter would drop it anyway,
# but only after we paid for SNAC encoding.
n_before = len(ds_train)
ds_train = ds_train.filter(
    lambda r: len(tokenizer.encode(r["text"], add_special_tokens=False)) <= MAX_SEQ_LEN
)
n_dropped = n_before - len(ds_train)
print(f"text pre-filter (text tokens <= {MAX_SEQ_LEN}): "
      f"{n_before} -> {len(ds_train)} ({n_dropped} dropped)")

snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cuda")

def tokenise_audio(waveform):
    waveform = torch.from_numpy(waveform).unsqueeze(0).to(dtype=torch.float32)
    waveform = waveform.unsqueeze(0).to("cuda")
    with torch.inference_mode():
        codes = snac_model.encode(waveform)
    all_codes = []
    for i in range(codes[0].shape[1]):
        all_codes.append(codes[0][0][i].item()       + 128266)
        all_codes.append(codes[1][0][2*i].item()     + 128266 + 4096)
        all_codes.append(codes[2][0][4*i].item()     + 128266 + 2*4096)
        all_codes.append(codes[2][0][(4*i)+1].item() + 128266 + 3*4096)
        all_codes.append(codes[1][0][(2*i)+1].item() + 128266 + 4*4096)
        all_codes.append(codes[2][0][(4*i)+2].item() + 128266 + 5*4096)
        all_codes.append(codes[2][0][(4*i)+3].item() + 128266 + 6*4096)
    return all_codes

def add_codes(example):
    codes_list = None
    try:
        answer_audio = example.get("audio")
        if answer_audio and "array" in answer_audio:
            codes_list = tokenise_audio(answer_audio["array"])
    except Exception as e:
        print(f"Skipping row due to error: {e}")
    example["codes_list"] = codes_list
    return example

ds_train = ds_train.map(add_codes, remove_columns=["audio"])

### #2.5 Drop empty/None code rows and remove duplicate frames

In [ ]:
ds_train = ds_train.filter(lambda x: x["codes_list"] is not None)
ds_train = ds_train.filter(lambda x: len(x["codes_list"]) > 0)

def remove_duplicate_frames(example):
    vals = example["codes_list"]
    if len(vals) % 7 != 0:
        raise ValueError("Input list length must be divisible by 7")
    result = vals[:7]
    for i in range(7, len(vals), 7):
        if vals[i] != result[-7]:
            result.extend(vals[i:i+7])
    example["codes_list"] = result
    return example

ds_train = ds_train.map(remove_duplicate_frames)
print(f"after dedupe: {len(ds_train)} rows")

### #2.6 Build training input_ids — multi-speaker tagged prompt format

The `create_input_ids` body is identical to the single-speaker notebook — `f"{example['source']}: {example['text']}"`. The only difference is that `example['source']` now varies per row (it's the row's own `speaker_id`, set in #2.2).

In [ ]:
TOKENISER_LENGTH   = 128256
END_OF_TEXT        = 128009
START_OF_SPEECH    = TOKENISER_LENGTH + 1   # 128257
END_OF_SPEECH      = TOKENISER_LENGTH + 2   # 128258
START_OF_HUMAN     = TOKENISER_LENGTH + 3   # 128259
END_OF_HUMAN       = TOKENISER_LENGTH + 4   # 128260
START_OF_AI        = TOKENISER_LENGTH + 5   # 128261
END_OF_AI          = TOKENISER_LENGTH + 6   # 128262
PAD_TOKEN          = TOKENISER_LENGTH + 7   # 128263
AUDIO_TOKENS_START = TOKENISER_LENGTH + 10  # 128266

def create_input_ids(example):
    text_prompt = f"{example['source']}: {example['text']}"
    text_ids = tokenizer.encode(text_prompt, add_special_tokens=True)
    text_ids.append(END_OF_TEXT)
    input_ids = (
        [START_OF_HUMAN]
        + text_ids
        + [END_OF_HUMAN]
        + [START_OF_AI]
        + [START_OF_SPEECH]
        + example["codes_list"]
        + [END_OF_SPEECH]
        + [END_OF_AI]
    )
    return {
        "input_ids": input_ids,
        "labels": input_ids,
        "attention_mask": [1] * len(input_ids),
    }

ds_train = ds_train.map(create_input_ids, remove_columns=["text", "codes_list"])
ds_train = ds_train.remove_columns(
    [c for c in ds_train.column_names if c not in ("input_ids", "labels", "attention_mask")]
)

# Drop rows whose tokenised length exceeds MAX_SEQ_LEN (the model's max
# context). Long Sunbird utterances + SNAC's ~80-frame/sec rate can
# blow past 2048 easily; without this filter trainer.train() crashes
# inside unsloth_fused_ce_loss when its auto-truncation hits a
# torch.compile graph break.
n_before = len(ds_train)
ds_train = ds_train.filter(lambda r: len(r["input_ids"]) <= MAX_SEQ_LEN)
n_dropped = n_before - len(ds_train)
print(f"length filter (<= {MAX_SEQ_LEN} tokens): {n_before} -> {len(ds_train)} "
      f"({n_dropped} dropped for being too long)")

print("train columns:", ds_train.column_names)
print("first row len:", len(ds_train[0]["input_ids"]))

### #2.7 Hold diverse test prompts aside — 1 per speaker, capped at 10

Pulls a held-out evaluation set spanning as many speakers / languages as possible, so #4 inference can demonstrate the model's per-speaker conditioning. Shuffles the test split first to avoid bias toward whichever language was concatenated first.

In [ ]:
MAX_TEST_PROMPTS = 10
ds_test_shuffled = ds_test.shuffle(seed=42)

test_prompts = []
_seen_speakers = set()
for row in ds_test_shuffled:
    sid = row["speaker_id"]
    if sid in _seen_speakers:
        continue
    _seen_speakers.add(sid)
    test_prompts.append({
        "speaker_id": sid,
        "language":   row["language"],
        "text":       row["text"],
    })
    if len(test_prompts) >= MAX_TEST_PROMPTS:
        break

print(f"held-aside test prompts ({len(test_prompts)} unique speakers):")
for p in test_prompts:
    print(f"  [{p['language']}/{p['speaker_id']}] {p['text'][:80]}")

## #3 Train

Default is a smoke run (`max_steps=60`). For a real multilingual fine-tune, set `max_steps=-1` and uncomment `num_train_epochs=3`. On RTX 4090 with the full Sunbird corpus, the real run is roughly 4–8 hours depending on row count. Consider raising `per_device_train_batch_size` to 4 (and dropping `grad_accum` to 1) for ~2× wall-clock speedup, since the 4090 has VRAM headroom.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Llama-style tokenizers ship without a pad token. Reuse Orpheus's
# PAD_TOKEN (128263) so the collator can pad variable-length input_ids /
# attention_mask cleanly when batch size > 1.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = PAD_TOKEN

# DataCollatorForSeq2Seq pads input_ids/attention_mask to the longest
# sequence in the batch and pads labels with -100 (ignored by loss).
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)

trainer = Trainer(
    model = model,
    train_dataset = ds_train,
    data_collator = data_collator,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60,                      # SMOKE DEFAULT
        max_steps = -1,                    # uncomment for the real run
        num_train_epochs = 3,              # uncomment for the real run
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = str(OUTPUT_DIR),
        report_to = "mlflow",
        save_total_limit = 2,                # cap disk usage
    ),
)

### GPU memory snapshot before training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

### Run training

In [ ]:
trainer_stats = trainer.train()

### GPU memory snapshot after training

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_for_lora = round(used_memory - start_gpu_memory, 3)
used_pct = round(used_memory / max_memory * 100, 3)
lora_pct = round(used_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_pct} %.")
print(f"Peak reserved memory for training % of max memory = {lora_pct} %.")

## #4 Inference

### #4.1 Use the diverse test_prompts from #2.7

Each entry has its own `speaker_id`, so the model gets prompted with `f"{speaker_id}: {text}"` per prompt — different voices for different prompts.

In [ ]:
all_prompts = test_prompts   # list of {speaker_id, language, text}
print(f"running inference on {len(all_prompts)} prompts:")
for p in all_prompts:
    print(f"  [{p['language']}/{p['speaker_id']}] {p['text'][:100]}")

### #4.2 Tokenise + pad + generate (per-prompt speaker)

In [ ]:
FastLanguageModel.for_inference(model)
snac_model.to("cpu")  # free GPU for the LM

prompts_tagged = [f"{p['speaker_id']}: {p['text']}" for p in all_prompts]
all_input_ids = [tokenizer(p, return_tensors="pt").input_ids for p in prompts_tagged]

start_token = torch.tensor([[128259]], dtype=torch.int64)         # SOH
end_tokens  = torch.tensor([[128009, 128260]], dtype=torch.int64) # EOT, EOH

modified = [torch.cat([start_token, ids, end_tokens], dim=1) for ids in all_input_ids]
max_len = max(t.shape[1] for t in modified)

padded, masks = [], []
for t in modified:
    pad = max_len - t.shape[1]
    padded.append(torch.cat([torch.full((1, pad), 128263, dtype=torch.int64), t], dim=1))
    masks.append(torch.cat([torch.zeros((1, pad), dtype=torch.int64),
                            torch.ones((1, t.shape[1]), dtype=torch.int64)], dim=1))

input_ids = torch.cat(padded, dim=0).to("cuda")
attention_mask = torch.cat(masks, dim=0).to("cuda")

generated_ids = model.generate(
    input_ids = input_ids,
    attention_mask = attention_mask,
    max_new_tokens = 1200,
    do_sample = True,
    temperature = 0.6,
    top_p = 0.95,
    repetition_penalty = 1.1,
    num_return_sequences = 1,
    eos_token_id = 128258,
    use_cache = True,
)

### #4.3 Crop, strip, redistribute SNAC codes, decode

In [ ]:
TOKEN_TO_FIND   = 128257                     # last SOS
AUDIO_TOKEN_LO  = 128266                     # first audio-codebook id
AUDIO_TOKEN_HI  = 128266 + 7 * 4096          # exclusive (156938)

indices = (generated_ids == TOKEN_TO_FIND).nonzero(as_tuple=True)
if len(indices[1]) > 0:
    last = indices[1][-1].item()
    cropped = generated_ids[:, last + 1:]
else:
    print("WARNING: no start-of-speech token (128257) in generated output."
          " Smoke runs (max_steps=60) often produce garbage — switch to the"
          " full-run epochs block in #3 for a real fine-tune.")
    cropped = generated_ids

rows = [row[(row >= AUDIO_TOKEN_LO) & (row < AUDIO_TOKEN_HI)] for row in cropped]

code_lists = []
for row in rows:
    n = (row.size(0) // 7) * 7
    if n == 0:
        print("WARNING: row has no usable audio tokens after filtering;"
              " emitting silence. (Smoke runs commonly hit this.)")
    trimmed = [t.item() - AUDIO_TOKEN_LO for t in row[:n]]
    code_lists.append(trimmed)

def redistribute_codes(code_list):
    layer_1, layer_2, layer_3 = [], [], []
    for i in range(len(code_list) // 7):
        layer_1.append(code_list[7*i])
        layer_2.append(code_list[7*i + 1] - 4096)
        layer_3.append(code_list[7*i + 2] - 2*4096)
        layer_3.append(code_list[7*i + 3] - 3*4096)
        layer_2.append(code_list[7*i + 4] - 4*4096)
        layer_3.append(code_list[7*i + 5] - 5*4096)
        layer_3.append(code_list[7*i + 6] - 6*4096)
    if not layer_1:
        return torch.zeros(1, 1, 12000)   # ~0.5s silence fallback
    def clamp(vals):
        return [max(0, min(4095, v)) for v in vals]
    codes = [torch.tensor(clamp(layer_1)).unsqueeze(0),
             torch.tensor(clamp(layer_2)).unsqueeze(0),
             torch.tensor(clamp(layer_3)).unsqueeze(0)]
    return snac_model.decode(codes)

samples = [redistribute_codes(cl) for cl in code_lists]

### #4.4 Display + save WAVs to `outputs/multilingual/inference_samples/`

Filename includes the speaker_id so multilingual outputs are easy to identify on disk.

In [ ]:
samples_dir = OUTPUT_DIR / "inference_samples"
samples_dir.mkdir(parents=True, exist_ok=True)

for idx, (p, audio_hat) in enumerate(zip(all_prompts, samples)):
    wav = audio_hat.detach().squeeze().to("cpu").numpy()
    sid = p["speaker_id"]
    out_path = samples_dir / f"sample_{idx:02d}_{sid}.wav"
    sf.write(str(out_path), wav, 24000)
    print(f"[{idx}] [{p['language']}/{sid}] {p['text'][:80]} -> {out_path}")
    display(AudioDisplay(wav, rate=24000))

## #5 Save & push

### #5.1 LoRA adapters (local)

In [ ]:
lora_dir = str(OUTPUT_DIR / "orpheus_lora")
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print("LoRA saved to", lora_dir)

### #5.2 Merged 16-bit checkpoint (local)

In [ ]:
merged_dir = str(OUTPUT_DIR / "orpheus_finetune_16bit")
model.save_pretrained_merged(
    merged_dir, tokenizer, save_method="merged_16bit",
)
print("merged 16-bit saved to", merged_dir)

### #5.2.1 Patch local `config.json` to match embedding shape

Unsloth's `save_pretrained_merged` runs a pad-token shim that adds `<|finetune_right_pad_id|>` to the tokenizer when `pad_token` is missing or equals `eos_token`. That bumps `len(tokenizer)` by 1 and Unsloth writes `config.vocab_size = len(tokenizer)`, but the merged 16-bit weights are not resized — so `config.vocab_size` ends up off-by-one vs. `embed_tokens.weight.shape[0]`. `transformers` tolerates this, but vLLM asserts equality and refuses to load. We re-derive `vocab_size` from the actual embedding tensor and rewrite `config.json`.

In [ ]:
import json
from pathlib import Path as _Path

_embed_size = model.get_input_embeddings().num_embeddings
_cfg_path = _Path(merged_dir) / "config.json"
_cfg = json.loads(_cfg_path.read_text())
if _cfg.get("vocab_size") != _embed_size:
    print(f"fix config.vocab_size: {_cfg.get('vocab_size')} -> {_embed_size}")
    _cfg["vocab_size"] = _embed_size
    _cfg_path.write_text(json.dumps(_cfg, indent=2))
else:
    print(f"config.vocab_size already matches embedding ({_embed_size})")

### #5.3 Push merged 16-bit to Hugging Face Hub

In [ ]:
HF_REPO_ID = "sunbird/orpheus-3b-tts-multilingual"
model.push_to_hub_merged(
    HF_REPO_ID, tokenizer,
    save_method="merged_16bit",
    token=os.environ["HF_TOKEN"],
)
print("pushed to https://huggingface.co/" + HF_REPO_ID)

### #5.3.1 Patch hub `config.json` to match embedding shape

`push_to_hub_merged` re-runs the same pad-token shim while writing to the hub, so the local fix in #5.2.1 doesn't propagate. Upload a corrected `config.json` directly so vLLM can load this repo without `hf_overrides`.

In [ ]:
from huggingface_hub import HfApi

_local_cfg = _Path(merged_dir) / "config.json"   # already fixed in #5.2.1
HfApi().upload_file(
    path_or_fileobj=str(_local_cfg),
    path_in_repo="config.json",
    repo_id=HF_REPO_ID,
    commit_message=f"fix(config): vocab_size -> {_embed_size} to match embed weights",
    token=os.environ["HF_TOKEN"],
)
print(f"hub config.json patched: vocab_size -> {_embed_size}")